<a href="https://colab.research.google.com/github/saadoonhammad/ieeecoins_data_imputation/blob/main/XGBoost_IEEE_COINS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 12.7 MB/s eta 0:00:00


**Hyperparameter Tuning - Limited Selected Dataset**

In [ ]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

input_folder_path = '/path/to/your/input_data'
output_folder_path = '/path/to/your/data/results'

param_dist = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': np.linspace(0.01, 0.2, 5),
    'subsample': np.linspace(0.6, 1.0, 5),
    'colsample_bytree': np.linspace(0.6, 1.0, 5)
}

results_dict = {}
file_patterns = ['c01m045e01', 'c05m124e01', 'c05m105e08']
# os.makedirs(output_folder_path, exist_ok=True)
comparison_results = []

for file_name in os.listdir(input_folder_path):
    if any(pattern in file_name for pattern in file_patterns) and file_name.endswith('.csv'):
        file_path = os.path.join(input_folder_path, file_name)
        print(f"\nProcessing file: {file_name}")
        # Determine the year based on the file name
        if '_2022' in file_name:
            year = '2022'
        elif '_2023' in file_name:
            year = '2023'
        else:
            year = '2021'  # Default year

        # Read the file
        data = pd.read_csv(file_path)
        data['timestamp'] = pd.to_datetime(data['timestamp'])
        data = data[['timestamp', 'temp_value_imp']]


        data['timestamp'] = data['timestamp'].apply(lambda x: x.replace(year=int(year)))
        data_filtered = data[(data['timestamp'] >= f'{year}-01-01') & (data['timestamp'] <= f'{year}-12-30')].reset_index(drop=True)

        data = pd.read_csv(file_path)
        # Feature Engineering: Add time-based features
        data_filtered['time_step'] = range(len(data_filtered))
        data_filtered['hour'] = data_filtered['timestamp'].dt.hour
        data_filtered['minute'] = data_filtered['timestamp'].dt.minute
        data_filtered['day_of_week'] = data_filtered['timestamp'].dt.dayofweek
        data_filtered['month'] = data_filtered['timestamp'].dt.month

        features = ['time_step', 'hour', 'minute', 'day_of_week', 'month']
        X = data_filtered[features]
        y = data_filtered['temp_value_imp']

        X = X.loc[y.notna()]
        y = y.loc[y.notna()]
        xgb_model = XGBRegressor(random_state=42)

        # Perform RandomizedSearchCV
        random_search = RandomizedSearchCV(
            estimator=xgb_model,
            param_distributions=param_dist,
            scoring='neg_root_mean_squared_error',
            n_iter=50,
            cv=5,
            verbose=1,
            random_state=42
        )

        random_search.fit(X, y)

        # Save Best Parameters and RMSE for this file
        best_params = random_search.best_params_
        best_rmse = np.sqrt(-random_search.best_score_)

        results_dict[file_name] = {
            "Best Parameters": best_params,
            "Best RMSE": best_rmse
        }

        print(f"Best RMSE for {file_name}: {best_rmse}")
        print(f"Best Parameters: {best_params}")

results_df = pd.DataFrame.from_dict(results_dict, orient='index')
# results_output_path = os.path.join(folder_path, "xgboost_tuning_results_all_data.csv")
# results_df.to_csv(results_output_path)

print("\n Hyperparameter tuning completed for all files!")




Processing file: c01m045e01_2021.csv
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best RMSE for c01m045e01_2021.csv: 2.237122792687971
Best Parameters: {'subsample': np.float64(0.6), 'n_estimators': 150, 'max_depth': 3, 'learning_rate': np.float64(0.0575), 'colsample_bytree': np.float64(0.6)}

Processing file: c05m124e01_2021.csv
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best RMSE for c05m124e01_2021.csv: 2.1015662056553825
Best Parameters: {'subsample': np.float64(0.9), 'n_estimators': 150, 'max_depth': 3, 'learning_rate': np.float64(0.105), 'colsample_bytree': np.float64(0.7)}

Processing file: c05m105e08_2022.csv
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best RMSE for c05m105e08_2022.csv: 2.166334558007655
Best Parameters: {'subsample': np.float64(0.7), 'n_estimators': 50, 'max_depth': 5, 'learning_rate': np.float64(0.0575), 'colsample_bytree': np.float64(0.6)}

Processing file: c05m105e08_2023.csv
Fitting 5 folds for each of

In [ ]:
results_df

,Best Parameters,Best RMSE
c01m045e01_2021.csv,"{'subsample': 0.6, 'n_estimators': 150, 'max_d...",2.237123
c05m124e01_2021.csv,"{'subsample': 0.9, 'n_estimators': 150, 'max_d...",2.101566
c05m105e08_2022.csv,"{'subsample': 0.7, 'n_estimators': 50, 'max_de...",2.166335
c05m105e08_2023.csv,"{'subsample': 0.6, 'n_estimators': 100, 'max_d...",2.312279
c05m124e01_2023.csv,"{'subsample': 0.9, 'n_estimators': 50, 'max_de...",2.308694
c01m045e01_2022.csv,"{'subsample': 0.6, 'n_estimators': 100, 'max_d...",2.299720
c01m045e01_2023.csv,"{'subsample': 0.7, 'n_estimators': 50, 'max_de...",2.514818
c05m124e01_2022.csv,"{'subsample': 0.6, 'n_estimators': 100, 'max_d...",2.104847
c05m105e08_2021.csv,"{'subsample': 0.6, 'n_estimators': 150, 'max_d...",2.161010


In [ ]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
import plotly.graph_objects as go

# Paths
input_folder_path = '/path/to/your/input_data'
output_folder_path = '/path/to/your/data/results'
file_patterns = ['c01m045e01', 'c05m124e01', 'c05m105e08']
#specify gap size
gap_sizes = [150, 300, 450, 600]
start_gap = 500

os.makedirs(output_folder_path, exist_ok=True)
comparison_results = []

for file_name in os.listdir(input_folder_path):
    if any(pattern in file_name for pattern in file_patterns) and file_name.endswith('.csv'):
        file_path = os.path.join(input_folder_path, file_name)

        if '_2022' in file_name:
            year = '2022'
        elif '_2023' in file_name:
            year = '2023'
        else:
            year = '2021'

        data = pd.read_csv(file_path)
        data['timestamp'] = pd.to_datetime(data['timestamp'])
        data = data[['timestamp', 'temp_value_imp']]
        data['timestamp'] = data['timestamp'].apply(lambda x: x.replace(year=int(year)))

        # Full range: June–September
        full_filtered = data[(data['timestamp'] >= f'{year}-06-01') & (data['timestamp'] <= f'{year}-09-30')].reset_index(drop=True)
        train_df = full_filtered[(full_filtered['timestamp'] >= f'{year}-06-01') & (full_filtered['timestamp'] <= f'{year}-08-31')].reset_index(drop=True)
        test_df_original = full_filtered[(full_filtered['timestamp'] >= f'{year}-09-01') & (full_filtered['timestamp'] <= f'{year}-09-30')].reset_index(drop=True)

        for gap_size in gap_sizes:
            test_df = test_df_original.copy()
            test_df.iloc[start_gap:start_gap + gap_size, test_df.columns.get_loc('temp_value_imp')] = np.nan

            for df in [train_df, test_df]:
                df['time_step'] = range(len(df))
                df['hour'] = df['timestamp'].dt.hour
                df['minute'] = df['timestamp'].dt.minute
                df['day_of_week'] = df['timestamp'].dt.dayofweek
                df['month'] = df['timestamp'].dt.month

            features = ['time_step', 'hour', 'minute', 'day_of_week', 'month']
            X_train = train_df[features]
            y_train = train_df['temp_value_imp']

            xgb_model = XGBRegressor(
                subsample=0.6,
                n_estimators=150,
                max_depth=3,
                learning_rate=0.0575,
                colsample_bytree=0.6,
                random_state=42
            )
            xgb_model.fit(X_train, y_train)

            known_data = test_df[test_df['temp_value_imp'].notna()]
            missing_data = test_df[test_df['temp_value_imp'].isna()]
            X_missing = missing_data[features]
            test_df.loc[missing_data.index, 'temp_value_imp'] = xgb_model.predict(X_missing)

            original_values = test_df_original.loc[missing_data.index, 'temp_value_imp']
            imputed_values = test_df.loc[missing_data.index, 'temp_value_imp']

            # Metrics
            rmse = root_mean_squared_error(original_values, imputed_values)
            mape = mean_absolute_percentage_error(original_values, imputed_values)
            mae = mean_absolute_error(original_values, imputed_values)

            comparison_results.append({
                'File Name': file_name,
                'Gap Size': gap_size,
                'RMSE': rmse,
                'MAPE': mape,
                'MAE': mae
            })

            # Save imputed vs original values
            result_df = pd.DataFrame({
                'timestamp': missing_data['timestamp'].values,
                'original_value': original_values.values,
                'imputed_value': imputed_values.values
            })
            result_filename = f"{file_name.split('.')[0]}_gap{gap_size}_xgb_imputed.csv"
            result_path = os.path.join(output_folder_path, result_filename)
            result_df.to_csv(result_path, index=False)

            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=test_df_original['timestamp'],
                y=test_df_original['temp_value_imp'],
                mode='lines',
                name='Original (Ground Truth)',
                line=dict(color='blue')
            ))
            fig.add_trace(go.Scatter(
                x=missing_data['timestamp'],
                y=imputed_values.values,
                mode='lines',
                name='Imputed Region',
                line=dict(color='red', width=2)
            ))
            fig.add_vrect(
                x0=missing_data['timestamp'].iloc[0],
                x1=missing_data['timestamp'].iloc[-1],
                fillcolor="gray",
                opacity=0.2,
                line_width=0,
                annotation_text="Missing Block",
                annotation_position="top left"
            )
            fig.update_layout(
                title=f'XGBoost Imputation | {file_name} | Gap Size: {gap_size} values',
                width=1000,
                height=500,
                xaxis_title="Timestamp",
                yaxis_title="Temperature",
                legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
            )

            plot_file_name = f"{file_name.split('.')[0]}_gap{gap_size}_xgb.png"
            plot_file_path = os.path.join(output_folder_path, plot_file_name)
            fig.write_image(plot_file_path, scale=2)

# Save comparison CSV
comparison_df = pd.DataFrame(comparison_results)
comparison_df["Year"] = comparison_df["File Name"].str.extract(r"_(\d{4})\.csv")
comparison_df["Suffix"] = comparison_df["File Name"].str.extract(r"e(\d{2})_")
comparison_df = comparison_df.sort_values(by=["Year", "Suffix", "Gap Size"])
comparison_df = comparison_df.drop(columns=["Year", "Suffix"])

comparison_results_file = os.path.join(output_folder_path, 'xgboost_imputation_results.csv')
comparison_df.to_csv(comparison_results_file, index=False)

print("\n✅ XGBoost imputation complete. All plots, CSVs, and imputed data saved.")



✅ XGBoost imputation complete. All plots, CSVs, and imputed data saved.


In [ ]:
comparison_df

,File Name,Gap Size,RMSE,MAPE,MAE
0,c01m045e01_2021.csv,150,2.574884,0.108820,2.131880
4,c05m124e01_2021.csv,150,2.254169,0.093058,2.105199
1,c01m045e01_2021.csv,300,4.163384,0.170357,3.600918
5,c05m124e01_2021.csv,300,3.485280,0.134287,3.199249
2,c01m045e01_2021.csv,450,4.886911,0.193186,4.211669
6,c05m124e01_2021.csv,450,3.808955,0.147209,3.555847
3,c01m045e01_2021.csv,600,4.329610,0.162936,3.537809
7,c05m124e01_2021.csv,600,3.644290,0.139399,3.383073
32,c05m105e08_2021.csv,150,2.400375,0.099380,2.061220
33,c05m105e08_2021.csv,300,3.416309,0.137698,3.000519


In [ ]:
fig.show()